# 15 Finetune Encoders

**Purpose.** Fine-tune the encoders end-to-end (seeded, stratified, class-weighted) as a comparison to freezing.

**Thesis reference.** §5 Representation, Not Model

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [1]:

import os, json, time, random
import numpy as np
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

IMAGE_ROOT  = "images224"
LABELS      = "labels/labels_severity_all.json"
IMAGE_TYPES = ["grayscale_heatmap", "rgb_mz_split"]
MODELS      = ["resnet18", "efficientnet_b0", "vit_b_16"]
EPOCHS      = 12
BATCH       = 16
LR          = 3e-4
N_SPLITS    = 5           # stratified CV instead of one 80/20 split
SEED        = 42
ID_BLOCK    = 0.582
CLASSICAL   = 0.741       # best classical model
FROZEN_BEST = 0.655       # best frozen-encoder result on this task

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMAGENET = ([0.485,0.456,0.406],[0.229,0.224,0.225])

lab = json.load(open(LABELS))["labels"]

def make_model(name, ch):
    if name == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        if ch == 1: m.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        if ch == 1:
            m.features[0][0] = nn.Conv2d(1,32,3,2,1,bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT)
        m.heads = nn.Linear(m.heads.head.in_features, 1)
        ch = 3
    return m, ch

class DS(Dataset):
    def __init__(self, folder, pids, ch, size):
        self.folder, self.pids, self.ch = Path(folder), pids, ch
        norm = transforms.Normalize(*IMAGENET) if ch == 3 else \
               transforms.Normalize([0.5],[0.5])
        self.tf = transforms.Compose([transforms.Resize((size,size)),
                                      transforms.ToTensor(), norm])
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        p = self.pids[i]
        mode = "RGB" if self.ch == 3 else "L"
        img = Image.open(self.folder/f"{p}.png").convert(mode)
        return self.tf(img), float(lab[p])

def run(image_type, model_name):
    folder = Path(IMAGE_ROOT)/image_type
    pids = [p for p in sorted(lab) if (folder/f"{p}.png").exists()]
    y = np.array([lab[p] for p in pids])
    size = 224
    base_ch = 3 if image_type in ("rgb_mz_split","pseudocolor_heatmap") else 1

    oof = np.zeros(len(pids))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(pids,y)):
        model, ch = make_model(model_name, base_ch)
        tr_pids = [pids[i] for i in tr]; te_pids = [pids[i] for i in te]
        dl_tr = DataLoader(DS(folder,tr_pids,ch,size), batch_size=BATCH, shuffle=True)
        dl_te = DataLoader(DS(folder,te_pids,ch,size), batch_size=BATCH)
        # class weight for the positive class
        pos = y[tr].mean(); w = torch.tensor([(1-pos)/max(pos,1e-6)])
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        for ep in range(EPOCHS):
            for xb,yb in dl_tr:
                opt.zero_grad()
                out = model(xb).squeeze(1)
                crit(out, yb).backward(); opt.step()
        model.eval(); preds=[]
        with torch.no_grad():
            for xb,_ in dl_te:
                preds.append(torch.sigmoid(model(xb).squeeze(1)).numpy())
        oof[te] = np.concatenate(preds)
        print(f"    fold {fold+1}/{N_SPLITS} done", end="\r", flush=True)
    return roc_auc_score(y, oof), len(pids)

print(f"fine-tuning on {LABELS}")
print(f"seed {SEED}, {N_SPLITS}-fold stratified, ImageNet norm, class-weighted loss\n")
print(f"{'image':20s} {'model':17s} {'AUC':>7} {'vs block':>9} {'vs frozen':>10} {'vs classical':>13}")
print("-"*80)
rows=[]
t0=time.time()
for it in IMAGE_TYPES:
    for mn in MODELS:
        try:
            auc,n = run(it,mn)
            print(f"{it:20s} {mn:17s} {auc:7.4f} {auc-ID_BLOCK:+9.4f} "
                  f"{auc-FROZEN_BEST:+10.4f} {auc-CLASSICAL:+13.4f}")
            rows.append((it,mn,auc))
        except Exception as e:
            print(f"{it:20s} {mn:17s} FAILED {type(e).__name__}: {e}")

print(f"\n{(time.time()-t0)/60:.0f} min")
if rows:
    b=max(rows,key=lambda r:r[2])
    print(f"\nbest fine-tuned : {b[0]} / {b[1]} = {b[2]:.4f}")
    print(f"best frozen     : {FROZEN_BEST:.4f}")
    print(f"best classical  : {CLASSICAL:.4f}")
    print(f"block baseline  : {ID_BLOCK:.4f}")

fine-tuning on labels/labels_severity_all.json
seed 42, 5-fold stratified, ImageNet norm, class-weighted loss

image                model                 AUC  vs block  vs frozen  vs classical
--------------------------------------------------------------------------------
grayscale_heatmap    resnet18           0.6715   +0.0895    +0.0165       -0.0695
grayscale_heatmap    efficientnet_b0    0.6123   +0.0303    -0.0427       -0.1287
grayscale_heatmap    vit_b_16           0.5152   -0.0668    -0.1398       -0.2258
rgb_mz_split         resnet18           0.6246   +0.0426    -0.0304       -0.1164
rgb_mz_split         efficientnet_b0    0.6316   +0.0496    -0.0234       -0.1094
rgb_mz_split         vit_b_16           0.5150   -0.0670    -0.1400       -0.2260

181 min

best fine-tuned : grayscale_heatmap / resnet18 = 0.6715
best frozen     : 0.6550
best classical  : 0.7410
block baseline  : 0.5820


In [2]:

import os, json, time, random
import numpy as np
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

IMAGE_ROOT  = "images"
LABELS      = "labels/labels_severity_all.json"
IMAGE_TYPES = ["grayscale_heatmap", "rgb_mz_split"]
MODELS      = ["resnet18", "efficientnet_b0", "vit_b_16"]
EPOCHS      = 12
BATCH       = 16
LR          = 3e-4
N_SPLITS    = 5           # stratified CV instead of one 80/20 split
SEED        = 42
ID_BLOCK    = 0.582
CLASSICAL   = 0.741       # best classical model
FROZEN_BEST = 0.655       # best frozen-encoder result on this task

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMAGENET = ([0.485,0.456,0.406],[0.229,0.224,0.225])

lab = json.load(open(LABELS))["labels"]

def make_model(name, ch):
    if name == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        if ch == 1: m.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        if ch == 1:
            m.features[0][0] = nn.Conv2d(1,32,3,2,1,bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT)
        m.heads = nn.Linear(m.heads.head.in_features, 1)
        ch = 3
    return m, ch

class DS(Dataset):
    def __init__(self, folder, pids, ch, size):
        self.folder, self.pids, self.ch = Path(folder), pids, ch
        norm = transforms.Normalize(*IMAGENET) if ch == 3 else \
               transforms.Normalize([0.5],[0.5])
        self.tf = transforms.Compose([transforms.Resize((size,size)),
                                      transforms.ToTensor(), norm])
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        p = self.pids[i]
        mode = "RGB" if self.ch == 3 else "L"
        img = Image.open(self.folder/f"{p}.png").convert(mode)
        return self.tf(img), float(lab[p])

def run(image_type, model_name):
    folder = Path(IMAGE_ROOT)/image_type
    pids = [p for p in sorted(lab) if (folder/f"{p}.png").exists()]
    y = np.array([lab[p] for p in pids])
    size = 224
    base_ch = 3 if image_type in ("rgb_mz_split","pseudocolor_heatmap") else 1

    oof = np.zeros(len(pids))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(pids,y)):
        model, ch = make_model(model_name, base_ch)
        tr_pids = [pids[i] for i in tr]; te_pids = [pids[i] for i in te]
        dl_tr = DataLoader(DS(folder,tr_pids,ch,size), batch_size=BATCH, shuffle=True)
        dl_te = DataLoader(DS(folder,te_pids,ch,size), batch_size=BATCH)
        # class weight for the positive class
        pos = y[tr].mean(); w = torch.tensor([(1-pos)/max(pos,1e-6)])
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        for ep in range(EPOCHS):
            for xb,yb in dl_tr:
                opt.zero_grad()
                out = model(xb).squeeze(1)
                crit(out, yb).backward(); opt.step()
        model.eval(); preds=[]
        with torch.no_grad():
            for xb,_ in dl_te:
                preds.append(torch.sigmoid(model(xb).squeeze(1)).numpy())
        oof[te] = np.concatenate(preds)
        print(f"    fold {fold+1}/{N_SPLITS} done", end="\r", flush=True)
    return roc_auc_score(y, oof), len(pids)

print(f"fine-tuning on {LABELS}")
print(f"seed {SEED}, {N_SPLITS}-fold stratified, ImageNet norm, class-weighted loss\n")
print(f"{'image':20s} {'model':17s} {'AUC':>7} {'vs block':>9} {'vs frozen':>10} {'vs classical':>13}")
print("-"*80)
rows=[]
t0=time.time()
for it in IMAGE_TYPES:
    for mn in MODELS:
        try:
            auc,n = run(it,mn)
            print(f"{it:20s} {mn:17s} {auc:7.4f} {auc-ID_BLOCK:+9.4f} "
                  f"{auc-FROZEN_BEST:+10.4f} {auc-CLASSICAL:+13.4f}")
            rows.append((it,mn,auc))
        except Exception as e:
            print(f"{it:20s} {mn:17s} FAILED {type(e).__name__}: {e}")

print(f"\n{(time.time()-t0)/60:.0f} min")
if rows:
    b=max(rows,key=lambda r:r[2])
    print(f"\nbest fine-tuned : {b[0]} / {b[1]} = {b[2]:.4f}")
    print(f"best frozen     : {FROZEN_BEST:.4f}")
    print(f"best classical  : {CLASSICAL:.4f}")
    print(f"block baseline  : {ID_BLOCK:.4f}")

fine-tuning on labels/labels_severity_all.json
seed 42, 5-fold stratified, ImageNet norm, class-weighted loss

image                model                 AUC  vs block  vs frozen  vs classical
--------------------------------------------------------------------------------
grayscale_heatmap    resnet18           0.4941   -0.0879    -0.1609       -0.2469
grayscale_heatmap    efficientnet_b0    0.5228   -0.0592    -0.1322       -0.2182
grayscale_heatmap    vit_b_16           0.5245   -0.0575    -0.1305       -0.2165
rgb_mz_split         resnet18           0.5416   -0.0404    -0.1134       -0.1994
rgb_mz_split         efficientnet_b0    0.4692   -0.1128    -0.1858       -0.2718
rgb_mz_split         vit_b_16           0.5089   -0.0731    -0.1461       -0.2321

238 min

best fine-tuned : rgb_mz_split / resnet18 = 0.5416
best frozen     : 0.6550
best classical  : 0.7410
block baseline  : 0.5820


In [3]:

import os, json, time, random
import numpy as np
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

IMAGE_ROOT  = "images_v2_occ_tic"
LABELS      = "labels/labels_severity_all.json"
IMAGE_TYPES = ["grayscale_heatmap", "rgb_mz_split"]
MODELS      = ["resnet18", "efficientnet_b0", "vit_b_16"]
EPOCHS      = 12
BATCH       = 16
LR          = 3e-4
N_SPLITS    = 5           # stratified CV instead of one 80/20 split
SEED        = 42
ID_BLOCK    = 0.582
CLASSICAL   = 0.741       # best classical model
FROZEN_BEST = 0.655       # best frozen-encoder result on this task

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMAGENET = ([0.485,0.456,0.406],[0.229,0.224,0.225])

lab = json.load(open(LABELS))["labels"]

def make_model(name, ch):
    if name == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        if ch == 1: m.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        if ch == 1:
            m.features[0][0] = nn.Conv2d(1,32,3,2,1,bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT)
        m.heads = nn.Linear(m.heads.head.in_features, 1)
        ch = 3
    return m, ch

class DS(Dataset):
    def __init__(self, folder, pids, ch, size):
        self.folder, self.pids, self.ch = Path(folder), pids, ch
        norm = transforms.Normalize(*IMAGENET) if ch == 3 else \
               transforms.Normalize([0.5],[0.5])
        self.tf = transforms.Compose([transforms.Resize((size,size)),
                                      transforms.ToTensor(), norm])
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        p = self.pids[i]
        mode = "RGB" if self.ch == 3 else "L"
        img = Image.open(self.folder/f"{p}.png").convert(mode)
        return self.tf(img), float(lab[p])

def run(image_type, model_name):
    folder = Path(IMAGE_ROOT)/image_type
    pids = [p for p in sorted(lab) if (folder/f"{p}.png").exists()]
    y = np.array([lab[p] for p in pids])
    size = 224
    base_ch = 3 if image_type in ("rgb_mz_split","pseudocolor_heatmap") else 1

    oof = np.zeros(len(pids))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(pids,y)):
        model, ch = make_model(model_name, base_ch)
        tr_pids = [pids[i] for i in tr]; te_pids = [pids[i] for i in te]
        dl_tr = DataLoader(DS(folder,tr_pids,ch,size), batch_size=BATCH, shuffle=True)
        dl_te = DataLoader(DS(folder,te_pids,ch,size), batch_size=BATCH)
        # class weight for the positive class
        pos = y[tr].mean(); w = torch.tensor([(1-pos)/max(pos,1e-6)])
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        for ep in range(EPOCHS):
            for xb,yb in dl_tr:
                opt.zero_grad()
                out = model(xb).squeeze(1)
                crit(out, yb).backward(); opt.step()
        model.eval(); preds=[]
        with torch.no_grad():
            for xb,_ in dl_te:
                preds.append(torch.sigmoid(model(xb).squeeze(1)).numpy())
        oof[te] = np.concatenate(preds)
        print(f"    fold {fold+1}/{N_SPLITS} done", end="\r", flush=True)
    return roc_auc_score(y, oof), len(pids)

print(f"fine-tuning on {LABELS}")
print(f"seed {SEED}, {N_SPLITS}-fold stratified, ImageNet norm, class-weighted loss\n")
print(f"{'image':20s} {'model':17s} {'AUC':>7} {'vs block':>9} {'vs frozen':>10} {'vs classical':>13}")
print("-"*80)
rows=[]
t0=time.time()
for it in IMAGE_TYPES:
    for mn in MODELS:
        try:
            auc,n = run(it,mn)
            print(f"{it:20s} {mn:17s} {auc:7.4f} {auc-ID_BLOCK:+9.4f} "
                  f"{auc-FROZEN_BEST:+10.4f} {auc-CLASSICAL:+13.4f}")
            rows.append((it,mn,auc))
        except Exception as e:
            print(f"{it:20s} {mn:17s} FAILED {type(e).__name__}: {e}")

print(f"\n{(time.time()-t0)/60:.0f} min")
if rows:
    b=max(rows,key=lambda r:r[2])
    print(f"\nbest fine-tuned : {b[0]} / {b[1]} = {b[2]:.4f}")
    print(f"best frozen     : {FROZEN_BEST:.4f}")
    print(f"best classical  : {CLASSICAL:.4f}")
    print(f"block baseline  : {ID_BLOCK:.4f}")

fine-tuning on labels/labels_severity_all.json
seed 42, 5-fold stratified, ImageNet norm, class-weighted loss

image                model                 AUC  vs block  vs frozen  vs classical
--------------------------------------------------------------------------------
grayscale_heatmap    resnet18           0.6630   +0.0810    +0.0080       -0.0780
grayscale_heatmap    efficientnet_b0    0.6135   +0.0315    -0.0415       -0.1275
grayscale_heatmap    vit_b_16           0.5236   -0.0584    -0.1314       -0.2174
rgb_mz_split         resnet18           0.6715   +0.0895    +0.0165       -0.0695
rgb_mz_split         efficientnet_b0    0.5956   +0.0136    -0.0594       -0.1454
rgb_mz_split         vit_b_16           0.5070   -0.0750    -0.1480       -0.2340

189 min

best fine-tuned : rgb_mz_split / resnet18 = 0.6715
best frozen     : 0.6550
best classical  : 0.7410
block baseline  : 0.5820


In [4]:

import os, json, time, random
import numpy as np
import torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
import torchvision.models as tvm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

IMAGE_ROOT  = "images_picked"
LABELS      = "labels/labels_severity_all.json"
IMAGE_TYPES = ["grayscale_heatmap", "rgb_mz_split"]
MODELS      = ["resnet18", "efficientnet_b0", "vit_b_16"]
EPOCHS      = 12
BATCH       = 16
LR          = 3e-4
N_SPLITS    = 5           # stratified CV instead of one 80/20 split
SEED        = 42
ID_BLOCK    = 0.582
CLASSICAL   = 0.741       # best classical model
FROZEN_BEST = 0.655       # best frozen-encoder result on this task

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMAGENET = ([0.485,0.456,0.406],[0.229,0.224,0.225])

lab = json.load(open(LABELS))["labels"]

def make_model(name, ch):
    if name == "resnet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        if ch == 1: m.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        if ch == 1:
            m.features[0][0] = nn.Conv2d(1,32,3,2,1,bias=False)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "vit_b_16":
        m = tvm.vit_b_16(weights=tvm.ViT_B_16_Weights.DEFAULT)
        m.heads = nn.Linear(m.heads.head.in_features, 1)
        ch = 3
    return m, ch

class DS(Dataset):
    def __init__(self, folder, pids, ch, size):
        self.folder, self.pids, self.ch = Path(folder), pids, ch
        norm = transforms.Normalize(*IMAGENET) if ch == 3 else \
               transforms.Normalize([0.5],[0.5])
        self.tf = transforms.Compose([transforms.Resize((size,size)),
                                      transforms.ToTensor(), norm])
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        p = self.pids[i]
        mode = "RGB" if self.ch == 3 else "L"
        img = Image.open(self.folder/f"{p}.png").convert(mode)
        return self.tf(img), float(lab[p])

def run(image_type, model_name):
    folder = Path(IMAGE_ROOT)/image_type
    pids = [p for p in sorted(lab) if (folder/f"{p}.png").exists()]
    y = np.array([lab[p] for p in pids])
    size = 224
    base_ch = 3 if image_type in ("rgb_mz_split","pseudocolor_heatmap") else 1

    oof = np.zeros(len(pids))
    skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
    for fold,(tr,te) in enumerate(skf.split(pids,y)):
        model, ch = make_model(model_name, base_ch)
        tr_pids = [pids[i] for i in tr]; te_pids = [pids[i] for i in te]
        dl_tr = DataLoader(DS(folder,tr_pids,ch,size), batch_size=BATCH, shuffle=True)
        dl_te = DataLoader(DS(folder,te_pids,ch,size), batch_size=BATCH)
        # class weight for the positive class
        pos = y[tr].mean(); w = torch.tensor([(1-pos)/max(pos,1e-6)])
        crit = nn.BCEWithLogitsLoss(pos_weight=w)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        for ep in range(EPOCHS):
            for xb,yb in dl_tr:
                opt.zero_grad()
                out = model(xb).squeeze(1)
                crit(out, yb).backward(); opt.step()
        model.eval(); preds=[]
        with torch.no_grad():
            for xb,_ in dl_te:
                preds.append(torch.sigmoid(model(xb).squeeze(1)).numpy())
        oof[te] = np.concatenate(preds)
        print(f"    fold {fold+1}/{N_SPLITS} done", end="\r", flush=True)
    return roc_auc_score(y, oof), len(pids)

print(f"fine-tuning on {LABELS}")
print(f"seed {SEED}, {N_SPLITS}-fold stratified, ImageNet norm, class-weighted loss\n")
print(f"{'image':20s} {'model':17s} {'AUC':>7} {'vs block':>9} {'vs frozen':>10} {'vs classical':>13}")
print("-"*80)
rows=[]
t0=time.time()
for it in IMAGE_TYPES:
    for mn in MODELS:
        try:
            auc,n = run(it,mn)
            print(f"{it:20s} {mn:17s} {auc:7.4f} {auc-ID_BLOCK:+9.4f} "
                  f"{auc-FROZEN_BEST:+10.4f} {auc-CLASSICAL:+13.4f}")
            rows.append((it,mn,auc))
        except Exception as e:
            print(f"{it:20s} {mn:17s} FAILED {type(e).__name__}: {e}")

print(f"\n{(time.time()-t0)/60:.0f} min")
if rows:
    b=max(rows,key=lambda r:r[2])
    print(f"\nbest fine-tuned : {b[0]} / {b[1]} = {b[2]:.4f}")
    print(f"best frozen     : {FROZEN_BEST:.4f}")
    print(f"best classical  : {CLASSICAL:.4f}")
    print(f"block baseline  : {ID_BLOCK:.4f}")

fine-tuning on labels/labels_severity_all.json
seed 42, 5-fold stratified, ImageNet norm, class-weighted loss

image                model                 AUC  vs block  vs frozen  vs classical
--------------------------------------------------------------------------------
grayscale_heatmap    resnet18           0.6578   +0.0758    +0.0028       -0.0832
grayscale_heatmap    efficientnet_b0    0.5642   -0.0178    -0.0908       -0.1768
grayscale_heatmap    vit_b_16           0.5065   -0.0755    -0.1485       -0.2345
rgb_mz_split         resnet18           0.6032   +0.0212    -0.0518       -0.1378
rgb_mz_split         efficientnet_b0    0.5986   +0.0166    -0.0564       -0.1424
rgb_mz_split         vit_b_16           0.5087   -0.0733    -0.1463       -0.2323

67 min

best fine-tuned : grayscale_heatmap / resnet18 = 0.6578
best frozen     : 0.6550
best classical  : 0.7410
block baseline  : 0.5820


In [5]:
import json
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

LABELS = "labels/labels_severity_all.json"
PROF   = "profiles"
ID_BLOCK = 0.582

lab  = json.load(open(LABELS))["labels"]
pids = json.load(open(f"{PROF}/profile_pids.json"))
idx  = [i for i,p in enumerate(pids) if p in lab]
y    = np.array([lab[pids[i]] for i in idx])
print(f"n = {len(y)}  ({int(y.sum())} mCKD / {int((1-y).sum())} eCKD)\n")

F = np.load(f"{PROF}/mz_profiles.npy")[idx].astype(np.float64)
C = np.load(f"{PROF}/coarse_maps.npy")[idx].astype(np.float64)
def norm(M):
    t=M.sum(1,keepdims=True); t[t==0]=1; return np.log1p(M/t*1e6)
REPS = {"profile_1d": norm(F), "map_2d": norm(C.reshape(len(C),-1))}

def sel(Xtr,Xte,k):
    if Xtr.shape[1]<=k: return Xtr,Xte
    o=np.argsort(Xtr.var(0))[::-1][:k]; return Xtr[:,o],Xte[:,o]

def ridge(): return Pipeline([("s",StandardScaler()),
    ("c",LogisticRegression(C=0.1,max_iter=4000,class_weight="balanced"))])
try:
    import xgboost as xgb
    def xgbm(): return xgb.XGBClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
        subsample=0.8,colsample_bytree=0.5,eval_metric="logloss",verbosity=0,n_jobs=4)
    HAVE=True
except ImportError: HAVE=False

def run(X,mk,k=3000):
    cv=RepeatedStratifiedKFold(n_splits=5,n_repeats=10,random_state=42)
    oof=np.zeros(len(y)); cnt=np.zeros(len(y))
    for tr,te in cv.split(X,y):
        A,B=sel(X[tr],X[te],k)
        oof[te]+=mk().fit(A,y[tr]).predict_proba(B)[:,1]; cnt[te]+=1
    o=oof/cnt; auc=roc_auc_score(y,o)
    rng=np.random.default_rng(0); bs=[]
    for _ in range(2000):
        i=rng.choice(len(y),len(y),replace=True)
        if len(np.unique(y[i]))>1: bs.append(roc_auc_score(y[i],o[i]))
    return auc, *np.percentile(bs,[2.5,97.5])

print(f"{'representation':14s} {'classifier':10s} {'AUC':>7} {'95% CI':>18}")
print("-"*54)
for rn,X in REPS.items():
    for cn,mk in ([("ridge",ridge)]+([("xgboost",xgbm)] if HAVE else [])):
        a,lo,hi=run(X,mk)
        print(f"{rn:14s} {cn:10s} {a:7.4f}  [{lo:.4f}, {hi:.4f}]")

print(f"\nblock baseline: {ID_BLOCK}")
# RepeatedStratifiedKFold(5 folds, 10 repeats, seed 42)


n = 510  (121 mCKD / 389 eCKD)

representation classifier     AUC             95% CI
------------------------------------------------------
profile_1d     ridge       0.5858  [0.5258, 0.6446]
profile_1d     xgboost     0.5728  [0.5150, 0.6266]
map_2d         ridge       0.7365  [0.6869, 0.7855]
map_2d         xgboost     0.7403  [0.6901, 0.7887]

block baseline: 0.582
